<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/correction_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation des dépendances

Installez (ou mettez à jour) les bibliothèques nécessaires pour ce notebook, en particulier **Keras** et **keras-hub**, directement depuis un notebook (commande shell).

In [ ]:
!pip install keras keras-hub --upgrade -q

### Configuration du backend Keras

Configurez l’environnement pour que Keras utilise le backend **TENSORFLOW** (via une variable d’environnement), avant d’importer Keras.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

### Construire un ConvNet pour MNIST

Avec Keras (API fonctionnelle), construisez un réseau de neurones convolutif pour classifier des images **28×28 en niveaux de gris** en **10 classes** :
- une entrée `Input(shape=(28, 28, 1))`,
- plusieurs couches `Conv2D` (activation ReLU),
- des couches de **max-pooling** pour réduire la taille spatiale,
- une couche de **global average pooling**,
- une couche `Dense(10, softmax)` en sortie.
Créez ensuite l’objet `Model(inputs, outputs)`.

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

### Inspecter l’architecture

Affichez le résumé (`summary`) du modèle en adaptant la largeur d’affichage pour que les lignes ne soient pas coupées.

In [ ]:
model.summary(line_length=80)

### Charger et entraîner sur MNIST

1. Chargez MNIST via `keras.datasets`.
2. Redimensionnez les tenseurs pour avoir un canal (forme `(N, 28, 28, 1)`).
3. Convertissez en `float32` et normalisez les pixels entre 0 et 1.
4. Compilez le modèle avec :
   - `optimizer="adam"`,
   - `loss="sparse_categorical_crossentropy"`,
   - `metrics=["accuracy"]`.
5. Entraînez pendant quelques epochs avec un `batch_size` raisonnable.

In [ ]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype("float32") / 255
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(train_images, train_labels, epochs=5, batch_size=64)

### Évaluation finale

Évaluez le modèle sur l’ensemble de test et affichez l’accuracy de test avec un formatage lisible (ex. 3 décimales).

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test accuracy: {test_acc:.3f}")

### Étudier l’effet du max-pooling

Construisez un second modèle **sans couche MaxPooling** :
- même entrée,
- plusieurs `Conv2D`,
- `GlobalAveragePooling2D`,
- `Dense(10, softmax)`.
Affichez ensuite son `summary` pour comparer les tailles d’activation / paramètres avec le modèle précédent.

In [ ]:
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(inputs)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model_no_max_pool = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model_no_max_pool.summary(line_length=80)

### Récupération des données (Kaggle)

Connectez-vous à Kaggle via `kagglehub`, puis téléchargez les données de la compétition **dogs-vs-cats** dans votre environnement.

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
download_path = kagglehub.competition_download("dogs-vs-cats")

### Décompression des données

À partir du chemin de téléchargement, décompressez l’archive `train.zip` afin d’obtenir le dossier contenant les images d’entraînement.

In [ ]:
import zipfile

with zipfile.ZipFile(download_path + "/train.zip", "r") as zip_ref:
    zip_ref.extractall(".")

### Construire un dataset "small" à partir du dataset initial

Créez une nouvelle arborescence de dossiers du type :

- `dogs_vs_cats_small/train/cat` et `dogs_vs_cats_small/train/dog`
- `dogs_vs_cats_small/validation/cat` et `dogs_vs_cats_small/validation/dog`
- `dogs_vs_cats_small/test/cat` et `dogs_vs_cats_small/test/dog`

Puis écrivez une fonction qui copie un intervalle d’images (par index dans le nom de fichier) depuis le dossier original vers chaque sous-ensemble (train/val/test).

In [ ]:
import os, shutil, pathlib

original_dir = pathlib.Path("train")
new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname, dst=dir / fname)

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

### Construire un ConvNet pour chiens vs chats

Avec l’API fonctionnelle Keras, construisez un modèle pour des images **180×180 RGB** :
- entrée `Input(shape=(180, 180, 3))`,
- normalisation via `Rescaling(1./255)`,
- plusieurs blocs `Conv2D + MaxPooling2D` (en augmentant progressivement le nombre de filtres),
- `GlobalAveragePooling2D`,
- sortie `Dense(1, sigmoid)` pour une classification binaire.
Affichez ensuite le `summary`.

In [ ]:
import keras
from keras import layers

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=512, kernel_size=3, activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [ ]:
model.summary(line_length=80)

### Compilation (classification binaire)

Compilez le modèle avec :
- `loss="binary_crossentropy"`,
- `optimizer="adam"`,
- `metrics=["accuracy"]`.

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

### Création des datasets Keras à partir de dossiers

À partir de l’arborescence `dogs_vs_cats_small`, créez :
- un dataset d’entraînement,
- un dataset de validation,
- un dataset de test,

en utilisant `image_dataset_from_directory`, avec :
- `image_size=(180, 180)`,
- `batch_size=64`.

In [ ]:
from keras.utils import image_dataset_from_directory

batch_size = 64
image_size = (180, 180)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=image_size, batch_size=batch_size
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=image_size, batch_size=batch_size
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=image_size, batch_size=batch_size
)

### Manipuler `tf.data.Dataset` (initiation)

1. Générez un tableau NumPy de nombres aléatoires (ex. forme `(1000, 16)`).
2. Transformez-le en `tf.data.Dataset` avec `from_tensor_slices`.
3. Itérez sur quelques éléments et affichez leurs formes.

In [ ]:
import numpy as np
import tensorflow as tf

random_numbers = np.random.normal(size=(1000, 16))
dataset = tf.data.Dataset.from_tensor_slices(random_numbers)

In [ ]:
for i, element in enumerate(dataset):
    print(element.shape)
    if i >= 2:
        break

### Batching

Regroupez le dataset en batchs (ex. 32) avec `.batch(32)` puis affichez la forme de quelques batchs.

In [ ]:
batched_dataset = dataset.batch(32)
for i, element in enumerate(batched_dataset):
    print(element.shape)
    if i >= 2:
        break

### Transformation avec `map`

Utilisez `.map()` pour transformer chaque élément du dataset en le redimensionnant (par exemple de `(16,)` vers `(4,4)`), en activant le parallélisme (`num_parallel_calls`).
Affichez ensuite la forme de quelques éléments transformés.

In [ ]:
reshaped_dataset = dataset.map(
    lambda x: tf.reshape(x, (4, 4)),
    num_parallel_calls=8)
for i, element in enumerate(reshaped_dataset):
    print(element.shape)
    if i >= 2:
        break

### Inspecter un batch du dataset d'entraînement

Parcourez le dataset d’entraînement et affichez :
- la forme d’un batch d’images,
- la forme du batch de labels,
puis arrêtez après le premier batch.

In [ ]:
for data_batch, labels_batch in train_dataset:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    break

### Entraînement avec sauvegarde automatique du meilleur modèle

Ajoutez un callback `ModelCheckpoint` pour sauvegarder automatiquement le **meilleur modèle** (selon `val_loss`) pendant l’entraînement.
Entraînez le modèle sur plusieurs epochs en utilisant :
- `train_dataset` pour l’entraînement,
- `validation_dataset` pour la validation,
- le callback de checkpointing.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="convnet_from_scratch.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks,
)

### Visualiser l'historique d'entraînement

À partir de l’objet `history` renvoyé par `fit`, récupérez :
- `accuracy` et `val_accuracy`,
- `loss` et `val_loss`,
puis tracez deux graphiques :
1. accuracy train vs val
2. loss train vs val

In [ ]:
import matplotlib.pyplot as plt

accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, "r--", label="Training accuracy")
plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()

plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

### Évaluation du meilleur modèle sauvegardé

Rechargez le modèle sauvegardé (fichier `.keras`) puis évaluez-le sur le dataset de test. Affichez l’accuracy de test avec un formatage clair.

In [ ]:
test_model = keras.models.load_model("convnet_from_scratch.keras")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

### Mettre en place une pipeline de data augmentation

1. Définissez une liste de couches d’augmentation (ex. flip horizontal, rotation, zoom).
2. Écrivez une fonction `data_augmentation(images, targets)` qui applique ces couches aux images.
3. Créez un dataset augmenté en appliquant cette fonction avec `.map(..., num_parallel_calls=...)`.
4. Ajoutez `.prefetch(tf.data.AUTOTUNE)` pour optimiser le pipeline.

In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

### Visualiser l’augmentation (9 images)

Affichez une grille 3×3 d’images augmentées à partir d’une même image du dataset, pour vérifier visuellement l’effet des transformations.

In [ ]:
plt.figure(figsize=(10, 10))
for image_batch, _ in train_dataset.take(1):
    image = image_batch[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image, _ = data_augmentation(image, None)
        augmented_image = keras.ops.convert_to_numpy(augmented_image)
        plt.imshow(augmented_image.astype("uint8"))
        plt.axis("off")

### Réentraîner avec augmentation + régularisation

Reprenez l’architecture précédente chiens/chats, ajoutez une couche `Dropout` avant la sortie, compilez, puis entraînez :
- sur `augmented_train_dataset`,
- avec validation sur `validation_dataset`,
- en sauvegardant le meilleur modèle (checkpoint sur `val_loss`).

In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=512, kernel_size=3, activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="convnet_from_scratch_with_augmentation.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
    callbacks=callbacks,
)

### Tester le modèle entraîné avec augmentation

Rechargez le modèle sauvegardé entraîné avec data augmentation puis évaluez-le sur le dataset de test. Affichez l’accuracy.

In [ ]:
test_model = keras.models.load_model(
    "convnet_from_scratch_with_augmentation.keras"
)
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

### Charger un modèle pré-entraîné

À l’aide de `keras_hub`, chargez un backbone pré-entraîné de type **Xception** (pré-entraîné sur ImageNet) qui servira d’extracteur de caractéristiques.

In [ ]:
import keras_hub

conv_base = keras_hub.models.Backbone.from_preset("xception_41_imagenet")

### Prétraitement compatible avec le modèle pré-entraîné

Chargez le préprocesseur / convertisseur d’images associé au preset choisi, en fixant `image_size=(180, 180)`.

In [ ]:
preprocessor = keras_hub.layers.ImageConverter.from_preset(
    "xception_41_imagenet",
    image_size=(180, 180),
)

### Extraire des caractéristiques (feature extraction)

Écrivez une fonction qui parcourt un dataset (images, labels) et :
1. applique le préprocesseur aux images,
2. calcule les features via le backbone pré-entraîné,
3. concatène toutes les features et tous les labels,
4. renvoie deux tableaux NumPy : `(features, labels)`.

Utilisez-la pour préparer train/val/test.

In [ ]:
def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        preprocessed_images = preprocessor(images)
        features = conv_base.predict(preprocessed_images, verbose=0)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

In [ ]:
train_features.shape

### Classifier sur features pré-calculées

Construisez un modèle Keras qui prend en entrée la forme des features extraites (ex. `(6, 6, 2048)`) et applique :
- `GlobalAveragePooling2D`,
- `Dense(256, relu)`,
- `Dropout`,
- `Dense(1, sigmoid)`.

Compilez (binary_crossentropy/adam/accuracy), entraînez sur `(train_features, train_labels)` avec validation sur `(val_features, val_labels)` et sauvegardez le meilleur modèle.

In [ ]:
inputs = keras.Input(shape=(6, 6, 2048))
x = layers.GlobalAveragePooling2D()(inputs)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    train_features,
    train_labels,
    epochs=10,
    validation_data=(val_features, val_labels),
    callbacks=callbacks,
)

### Courbes d’apprentissage (feature extraction)

Tracez accuracy et loss (train vs validation) pour l’entraînement du classifieur sur features, à partir de `history`.

In [ ]:
import matplotlib.pyplot as plt

acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, "r--", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

### Test final (features pré-calculées)

Rechargez le meilleur modèle sauvegardé et évaluez-le sur `test_features` / `test_labels`, puis affichez l’accuracy.

In [ ]:
test_model = keras.models.load_model("feature_extraction.keras")
test_loss, test_acc = test_model.evaluate(test_features, test_labels)
print(f"Test accuracy: {test_acc:.3f}")

### Backbone gelé

Rechargez le backbone Xception en précisant qu’il n’est pas entraînable (`trainable=False`) afin de l’utiliser comme extracteur fixe.

In [ ]:
import keras_hub

conv_base = keras_hub.models.Backbone.from_preset(
    "xception_41_imagenet",
    trainable=False,
)

### Vérifier les poids entraînables

Passez temporairement le backbone en `trainable=True` puis en `trainable=False`, et affichez le nombre de poids entraînables (`len(model.trainable_weights)`) pour vérifier que le gel fonctionne.

In [ ]:
conv_base.trainable = True
len(conv_base.trainable_weights)

In [ ]:
conv_base.trainable = False
len(conv_base.trainable_weights)

### Modèle end-to-end avec backbone gelé

Construisez un modèle Keras qui :
- prend des images 180×180×3 en entrée,
- applique le préprocesseur,
- passe dans le backbone,
- fait `GlobalAveragePooling2D`,
- ajoute une petite tête dense (Dense + Dropout),
- sort une probabilité binaire (Dense(1, sigmoid)).

Compilez ensuite le modèle (binary_crossentropy/adam/accuracy).

In [ ]:
inputs = keras.Input(shape=(180, 180, 3))
x = preprocessor(inputs)
x = conv_base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

### Entraînement avec augmentation (backbone gelé)

Entraînez ce modèle sur `augmented_train_dataset`, validez sur `validation_dataset`, et sauvegardez le meilleur modèle (monitor `val_loss`).

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction_with_data_augmentation.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    augmented_train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks,
)

### Test du modèle avec backbone gelé + augmentation

Rechargez le meilleur modèle sauvegardé et évaluez-le directement sur `test_dataset`. Affichez l’accuracy.

In [ ]:
test_model = keras.models.load_model(
    "feature_extraction_with_data_augmentation.keras"
)
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

### Fine-tuning du modèle pré-entraîné

Re-compilez le modèle en utilisant un optimiseur Adam avec un **faible learning rate** (ex. `1e-5`), puis relancez un entraînement sur `augmented_train_dataset` avec validation, en sauvegardant le meilleur modèle (sur `val_loss`).

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    augmented_train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks,
)

### Évaluer le modèle fine-tuné

Rechargez le modèle fine-tuné sauvegardé et évaluez-le sur `test_dataset`. Affichez l’accuracy.

In [ ]:
model = keras.models.load_model("fine_tuning.keras")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")